# Stages 2-4: mechanistic kinetic analysis 

This notebook reads a `*_fulldata.xlsx` workbook, calculates the concentration of the free form of the nucleophile at the reaction pH (e.g., in case of amines, the free base), performs the sequential $k_{\mathrm{obs}} \rightarrow k_{2,\mathrm{app}} \rightarrow$ pH-rate analysis (Strategy A), and performs a direct global fit from $k_{\mathrm{obs}}$ to the pH-rate model (Strategy B). The notebook is meant to be placed in the main kinetics directory alongside the `P5`, `P6`, ..., directories. Enter the name of one phosphorylating-agent directory in `KINETICS_DIRECTORY`; all of its `S*` dataset subdirectories are discovered automatically.

## Analysis overview

The analysis starts from the curated `*_fulldata.xlsx` files generated in Stage 1a and Stage 1b of the kinetic workflow. Each row corresponds to one kinetic measurement at a defined total nucleophile concentration, pH, and electrophile identity. The notebook converts total nucleophile concentration into the concentration of the reactive/free nucleophile, selects suitable $k_{\mathrm{obs}}$ values, and then estimates second-order rate constants.

Two complementary strategies are implemented:

1. **Strategy A, sequential analysis.** For each nucleophile and nominal buffer pH, $k_{\mathrm{obs}}$ is regressed against free nucleophile concentration. The slope is the apparent second-order rate constant $k_{2,\mathrm{app}}$; the intercept is the pH-specific background rate $k_0$. Successful $k_{2,\mathrm{app}}$ values are then fitted as a function of corrected pH.
2. **Strategy B, global analysis.** All selected $k_{\mathrm{obs}}$ values for one nucleophile/electrophile pair are fitted at once. This directly estimates the pH-dependent $k_{2,\mathrm{app}}$ profile while allowing a separate $k_0$ for each pH series.

All nucleophile-specific plot titles report the nucleophile name followed by its bold `Nu` identifier in parentheses.


## Configuration

Set `KINETICS_DIRECTORY` in the configuration cell to the directory that should be analyzed, for example:

`KINETICS_DIRECTORY = "P7"`

The notebook analyzes every `S*` dataset directory inside the selected directory. No directory is selected automatically. Outputs are written into a `k2_next_step_analysis` folder inside each dataset directory.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.optimize import least_squares

warnings.filterwarnings("ignore", category=RuntimeWarning)

# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------
NOTEBOOK_DIR = Path.cwd().resolve()
KINETICS_DIRECTORY = "P7"  # Enter one directory name, for example "P7"

KINETICS_DIRECTORY = KINETICS_DIRECTORY.strip()
if not KINETICS_DIRECTORY:
    raise ValueError("Set KINETICS_DIRECTORY to the directory that should be analyzed, for example 'P7'.")
directory_name = Path(KINETICS_DIRECTORY)
if directory_name.is_absolute() or directory_name.name != KINETICS_DIRECTORY:
    raise ValueError("KINETICS_DIRECTORY must contain one directory name, not a path.")
BASE_DIR = (NOTEBOOK_DIR / directory_name).resolve()
if not BASE_DIR.is_dir():
    raise FileNotFoundError(f"Kinetics directory not found: {BASE_DIR}")

PKA_PATH = NOTEBOOK_DIR / "phosphorylating_pka.xlsx"
NUCLEOPHILE_PATH = NOTEBOOK_DIR / "nucleophile_pka.xlsx"
DATASET_DIRS = [path.name for path in sorted(BASE_DIR.glob("S*")) if path.is_dir()]
if not DATASET_DIRS:
    raise FileNotFoundError(f"No dataset directories were found in {BASE_DIR}")

BOOTSTRAP_N = 2000  # Bayesian bootstrap count
RANDOM_SEED = 57

MIN_POINTS = 3  # Minimum number of observations required for fitting
MIN_DISTINCT_NU = 3  # Minimum number of distinct nucleophile concentrations
STRICT_R2_REFERENCE_THRESHOLD = 0.90  # R-squared cutoff for the strict reference flag
K2_APP_ACCEPT_MIN_R2 = 0.50  # Minimum R-squared for accepting a k2,app fit
K2_APP_ACCEPT_MAX_REL_CI_WIDTH = 1.00  # Maximum relative 95% CI width for accepting a k2,app fit
OUTLIER_FACTOR = 3.0  # Robust-residual multiplier used to identify outliers
                      # (Stage 2 linear fit, Stage 3 pH-rate fit, Stage 4 global fit)
PREFERRED_METHODS = {"exponential", "initial_rate_exp_derivative"}  # Preferred methods for estimating k_obs
FALLBACK_METHOD = "initial_rate_linear_fallback"  # Backup k_obs method when preferred fits are insufficient
EXCLUDED_METHODS = {"man_removed"}  # Methods omitted from mechanistic fitting

OUTPUT_SUBDIR = "k2_next_step_analysis"
SAVE_KOBS_PLOTS = True

rng = np.random.default_rng(RANDOM_SEED)

plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print(f"BASE_DIR = {BASE_DIR}")
print(f"PKA_PATH = {PKA_PATH}")
print(f"NUCLEOPHILE_PATH = {NUCLEOPHILE_PATH}")
print(f"DATASET_DIRS = {DATASET_DIRS}")
print(f"BOOTSTRAP_N = {BOOTSTRAP_N}")

## Helper functions and reference values

This section defines small numerical and plotting helpers used throughout the notebook. It loads the electrophile pKa table from `phosphorylating_pka.xlsx` and the authoritative nucleophile names and `Nu` identifiers from `nucleophile_pka.xlsx`, both located beside this notebook. The pKa value of a phosphorylating agent is used to calculate the fraction of the electrophile in the high-pH/deprotonated state:

$$
\alpha = \frac{1}{1 + 10^{pK_a - pH}}
$$

and the fitted pH-rate model is:

$$
k_{2,\mathrm{app}}(pH) = (1-\alpha)k_{2,\mathrm{low}} + \alpha k_{2,\mathrm{high}}
$$

In [ ]:
# ---------------------------------------------------------------------
# General helpers
# ---------------------------------------------------------------------
def clean_method(value):
    if pd.isna(value):
        return ""
    return str(value).strip()


def finite_positive_mask(*arrays):
    mask = np.ones(len(arrays[0]), dtype=bool)
    for arr in arrays:
        arr = np.asarray(arr, dtype=float)
        mask &= np.isfinite(arr)
    return mask


def alpha_deprotonated(pH, pKa):
    '''Fraction of electrophile or nucleophile in the deprotonated/high-pH state.'''
    pH = np.asarray(pH, dtype=float)
    return 1.0 / (1.0 + 10.0 ** (pKa - pH))


def r2_score(y_obs, y_pred):
    y_obs = np.asarray(y_obs, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    ss_res = np.sum((y_obs - y_pred) ** 2)
    ss_tot = np.sum((y_obs - np.mean(y_obs)) ** 2)
    return np.nan if ss_tot <= 0 else 1.0 - ss_res / ss_tot


def ci(values):
    values = np.asarray(values, dtype=float)
    values = values[np.all(np.isfinite(values), axis=-1)] if values.ndim > 1 else values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan
    lo, hi = np.percentile(values, [2.5, 97.5], axis=0)
    return lo, hi


def safe_log10(values, tiny=1e-300):
    return np.log10(np.maximum(np.asarray(values, dtype=float), tiny))


def fmt_sci(value, digits=2):
    if value is None or not np.isfinite(value):
        return "n/a"
    return f"{value:.{digits}e}"


def fmt_float(value, digits=3):
    if value is None or not np.isfinite(value):
        return "n/a"
    return f"{value:.{digits}f}"


def sanitize_filename(text):
    text = str(text)
    keep = []
    for ch in text:
        if ch.isalnum() or ch in ("-", "_", "."):
            keep.append(ch)
        elif ch.isspace():
            keep.append("_")
    return "".join(keep).strip("_") or "unnamed"


def format_nucleophile_label(nu_name, nu_id):
    '''Return a publication-style label with the nucleophile ID in bold.'''
    name = "" if pd.isna(nu_name) else str(nu_name).strip()
    identifier = "" if pd.isna(nu_id) else str(nu_id).strip()
    bold_identifier = f"$\\mathbf{{{identifier}}}$" if identifier else ""
    if name and bold_identifier:
        return f"{name} ({bold_identifier})"
    return name or bold_identifier or "unknown nucleophile"


def output_dir_for(dataset_dir):
    out = dataset_dir / OUTPUT_SUBDIR
    out.mkdir(parents=True, exist_ok=True)
    (out / "plots").mkdir(exist_ok=True)
    (out / "kobs_vs_nu_plots").mkdir(exist_ok=True)
    return out


def find_fulldata(dataset_dir):
    files = sorted(dataset_dir.glob("*fulldata.xlsx"))
    if len(files) != 1:
        raise FileNotFoundError(f"Expected exactly one *fulldata.xlsx in {dataset_dir}, found {len(files)}: {files}")
    return files[0]


def load_pka_table(pka_path):
    if not pka_path.is_file():
        raise FileNotFoundError(
            f"Could not find {pka_path}. Place phosphorylating_pka.xlsx beside this notebook."
        )
    df = pd.read_excel(pka_path)
    df.columns = [str(c).strip() for c in df.columns]
    px_col = "PX" if "PX" in df.columns else df.columns[0]
    pka_col = "pKa" if "pKa" in df.columns else df.columns[1]
    table = dict(zip(df[px_col].astype(str).str.strip(), pd.to_numeric(df[pka_col], errors="coerce")))
    print(f"Loaded electrophile pKa values from {pka_path}")
    return table


NUCLEOPHILE_NAME_ALIASES = {
    "4-chlorphenol": "4-chlorophenol",
    "4-cyanopyridin": "4-cyanopyridine",
    "4-methylpiyridine": "4-methylpyridine",
    "methylimidazole": "n-methylimidazole",
    "n-acetyl-l-cysteine": "n-acetylcysteine",
}


def nucleophile_name_key(value):
    if pd.isna(value):
        return ""
    text = str(value).strip().lower().replace("_", " ")
    text = NUCLEOPHILE_NAME_ALIASES.get(text, text)
    return "".join(ch for ch in text if ch.isalnum())


def load_nucleophile_table(path):
    if not path.is_file():
        raise FileNotFoundError(
            f"Could not find {path}. Place nucleophile_pka.xlsx beside this notebook."
        )
    reference = pd.read_excel(path)
    required = {"Nu_ID", "Nucleophile name"}
    missing = required - set(reference.columns)
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {sorted(missing)}")
    reference = reference[["Nu_ID", "Nucleophile name"]].dropna().copy()
    reference["Nu_ID"] = reference["Nu_ID"].astype(str).str.strip()
    reference["Nucleophile name"] = reference["Nucleophile name"].astype(str).str.strip()
    reference["name_key"] = reference["Nucleophile name"].map(nucleophile_name_key)
    if reference["Nu_ID"].duplicated().any() or reference["name_key"].duplicated().any():
        raise ValueError(f"{path.name} contains duplicate nucleophile identifiers or names")
    print(f"Loaded nucleophile names and identifiers from {path}")
    return reference


P_ELECTROPHILE_PKA = load_pka_table(PKA_PATH)
NUCLEOPHILE_REFERENCE = load_nucleophile_table(NUCLEOPHILE_PATH)
NUCLEOPHILE_BY_NAME = NUCLEOPHILE_REFERENCE.set_index("name_key").to_dict(orient="index")
NUCLEOPHILE_BY_ID = {row["Nu_ID"].lower(): row for row in NUCLEOPHILE_REFERENCE.to_dict(orient="records")}
P_ELECTROPHILE_PKA

## Data preparation

The required input workbook is detected by the `*fulldata.xlsx` filename pattern. The notebook uses the corrected-pH column `pH_corr_ionize`, assigns canonical nucleophile names and `Nu` identifiers from `nucleophile_pka.xlsx`, and checks that the required columns are present.

The reactive/free nucleophile concentration is calculated from total nucleophile concentration, corrected pH, nucleophile pKa, and nucleophile charge annotation. In the current implementation, the reactive form is treated as the deprotonated form:

$$
[Nu]_{free} = [Nu]_{tot}\frac{1}{1 + 10^{pK_{a,Nu} - pH_{corr}}}
$$

For `nu_chrg == 1`, this corresponds to the neutral deprotonated nucleophile. For `nu_chrg == 0`, it corresponds to the anionic deprotonated nucleophile. Rows marked as manually removed are excluded from fitting.

In [ ]:
# ---------------------------------------------------------------------
# Data preparation and method selection
# ---------------------------------------------------------------------
REQUIRED_COLUMNS = [
    "expnr", "entry", "electrophile_type", "nu_id", "nu_name",
    "ha_initial_pH", "pH_corr_ionize", "nu_pka_corr", "nu_chrg",
    "[nu]tot", "k_obs", "k_obs_method",
]

def standardize_nucleophile_columns(df):
    out = df.copy()
    resolved = []
    unresolved = set()
    for nu_name, nu_id in zip(out["nu_name"], out["nu_id"]):
        record = None
        for candidate in (nu_name, nu_id):
            key = nucleophile_name_key(candidate)
            if key in NUCLEOPHILE_BY_NAME:
                record = NUCLEOPHILE_BY_NAME[key]
                break
            identifier = "" if pd.isna(candidate) else str(candidate).strip().lower()
            if identifier in NUCLEOPHILE_BY_ID:
                record = NUCLEOPHILE_BY_ID[identifier]
                break
        if record is None:
            unresolved.add(str(nu_name if not pd.isna(nu_name) else nu_id))
            resolved.append((np.nan, np.nan))
        else:
            resolved.append((record["Nu_ID"], record["Nucleophile name"]))
    if unresolved:
        raise ValueError("Nucleophiles missing from nucleophile_pka.xlsx: " + ", ".join(sorted(unresolved)))
    out[["nu_id", "nu_name"]] = pd.DataFrame(resolved, index=out.index)
    return out


def add_free_nucleophile(df):
    '''Calculate active/free nucleophile concentration from corrected pH and nucleophile pKa.

    nu_chrg == 1: pKa refers to a protonated cation; neutral deprotonated nucleophile is active.
    nu_chrg == 0: pKa refers to neutral HA; anionic deprotonated nucleophile is active.
    Other charge states are still treated as the deprotonated form, but flagged.
    '''
    out = df.copy()
    pH = pd.to_numeric(out["pH_corr_ionize"], errors="coerce")
    pKa = pd.to_numeric(out["nu_pka_corr"], errors="coerce")
    total = pd.to_numeric(out["[nu]tot"], errors="coerce")
    charge = pd.to_numeric(out["nu_chrg"], errors="coerce")

    frac = alpha_deprotonated(pH, pKa)
    out["nu_active_fraction"] = frac
    out["nu_free"] = total * frac
    out["nu_active_charge_interpretation"] = np.select(
        [charge == 1, charge == 0],
        ["neutral deprotonated nucleophile", "anionic deprotonated nucleophile"],
        default="deprotonated nucleophile, charge state not explicitly classified",
    )
    return out


def load_dataset(dataset_dir):
    fulldata = find_fulldata(dataset_dir)
    df = pd.read_excel(fulldata)
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"{fulldata.name} is missing required columns: {missing}")
    df = standardize_nucleophile_columns(df)
    df = add_free_nucleophile(df)
    df["k_obs_method_clean"] = df["k_obs_method"].map(clean_method)
    df["k_obs"] = pd.to_numeric(df["k_obs"], errors="coerce")
    df["ha_initial_pH"] = pd.to_numeric(df["ha_initial_pH"], errors="coerce")
    df["pH_corr_ionize"] = pd.to_numeric(df["pH_corr_ionize"], errors="coerce")
    df["electrophile_type"] = df["electrophile_type"].astype(str).str.strip()
    return df, fulldata


def select_rows_for_group(group):
    eligible = group[
        (~group["k_obs_method_clean"].isin(EXCLUDED_METHODS))
        & np.isfinite(group["k_obs"])
        & (group["k_obs"] > 0)
        & np.isfinite(group["nu_free"])
    ].copy()
    if eligible.empty:
        return eligible, "no usable kobs"

    preferred = eligible[eligible["k_obs_method_clean"].isin(PREFERRED_METHODS)]
    fallback = eligible[eligible["k_obs_method_clean"].eq(FALLBACK_METHOD)]

    if len(preferred) >= MIN_POINTS and preferred["nu_free"].nunique() >= MIN_DISTINCT_NU:
        return preferred.copy(), "preferred only"

    mixed = pd.concat([preferred, fallback], axis=0).sort_index()
    if len(preferred) > 0 and len(mixed) >= MIN_POINTS and mixed["nu_free"].nunique() >= MIN_DISTINCT_NU:
        return mixed.copy(), "preferred plus linear fallback"

    # Last resort: all non-manual-removal methods, still requiring enough concentration leverage.
    if len(eligible) >= MIN_POINTS and eligible["nu_free"].nunique() >= MIN_DISTINCT_NU:
        return eligible.copy(), "last-resort mixed methods"

    return eligible.copy(), "insufficient usable points"


def eligible_rows_for_group(group):
    return group[
        (~group["k_obs_method_clean"].isin(EXCLUDED_METHODS))
        & np.isfinite(group["k_obs"])
        & (group["k_obs"] > 0)
        & np.isfinite(group["nu_free"])
    ].copy()

## Stage 2: extraction of apparent second-order rate constants

For each electrophile, nucleophile, and `ha_initial_pH` group, the notebook fits:

$$
k_{obs} = k_0 + k_{2,\mathrm{app}}[Nu]_{free}
$$

The method-selection rule is conservative:

- Test `exponential` alone.
- Test `initial_rate_exp_derivative` alone.
- Test the combined preferred set (`exponential + initial_rate_exp_derivative`).
- Only then test `initial_rate_linear_fallback` and mixed fallback candidates.
- Select the first candidate, in that priority order, that passes the $k_{2,\mathrm{app}}$ acceptance criteria below.

Outlier handling uses a two-step Huber/MAD workflow described in the SI. First, a Huber fit estimates residuals; then the residual median and MAD define the outlier cutoff. Points with `abs(residual - median residual) > 3 * 1.4826 * MAD` are treated as outliers for the final fit. Bayesian bootstrapping gives confidence intervals for $k_{2,\mathrm{app}}$ and $k_0$.

A $k_{2,\mathrm{app}}$ value is considered successful only if $k_{2,\mathrm{app}} > 0$, its bootstrap lower 95% confidence limit is positive, the inlier fit has $R^2 \geq 0.50$, and its relative 95% confidence-interval width is no larger than 1.0. Failed groups are retained in the Excel sheet for auditability, but they are not used in the Strategy A pH-rate fit.

Outlier detection is always applied **within the candidate subset being tested**. For example, if `exponential only` is tested, only exponential-derived points can be inliers or outliers in that candidate. If `exponential + initial_rate_exp_derivative` is tested, both sources are considered together and the Huber/MAD residual rule decides outliers within that mixed candidate. Points from methods that are not part of the selected candidate are shown as method-excluded points in the PNGs, not as statistical outliers.

In [ ]:
# ---------------------------------------------------------------------
# Strategy A, first stage: kobs vs free nucleophile concentration
# ---------------------------------------------------------------------
def linear_predict(params, x):
    intercept, slope = params
    return intercept + slope * x


def linear_residuals(params, x, y, weights=None):
    resid = linear_predict(params, x) - y
    if weights is not None:
        resid = resid * np.sqrt(weights)
    return resid


def fit_linear_two_step_huber(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = finite_positive_mask(x, y)
    x, y = x[mask], y[mask]
    if len(x) < MIN_POINTS or len(np.unique(x)) < MIN_DISTINCT_NU:
        return None

    def huber_fit(x_fit, y_fit, p0, f_scale):
        return least_squares(
            linear_residuals, x0=p0, args=(x_fit, y_fit),
            loss="huber", f_scale=max(f_scale, 1e-30),
            ftol=1e-12, xtol=1e-12, gtol=1e-12
        )

    slope0, intercept0 = np.polyfit(x, y, deg=1)
    p0 = np.array([intercept0, slope0], dtype=float)
    span = max(np.std(y), 1e-30)
    res0 = huber_fit(x, y, p0, span)
    resid0 = linear_residuals(res0.x, x, y)
    resid_median = np.median(resid0)
    mad0 = np.median(np.abs(resid0 - resid_median))
    f_scale = max(1.4826 * mad0, 1e-30)
    inlier_mask = np.abs(resid0 - resid_median) <= OUTLIER_FACTOR * f_scale
    if inlier_mask.sum() < MIN_POINTS or len(np.unique(x[inlier_mask])) < MIN_DISTINCT_NU:
        inlier_mask = np.ones(len(x), dtype=bool)

    resid_in = linear_residuals(res0.x, x[inlier_mask], y[inlier_mask])
    mad_in = np.median(np.abs(resid_in - np.median(resid_in)))
    final_scale = max(1.4826 * mad_in, span * 1e-3, 1e-30)
    res1 = huber_fit(x[inlier_mask], y[inlier_mask], res0.x, final_scale)
    pred_in = linear_predict(res1.x, x[inlier_mask])
    r2 = r2_score(y[inlier_mask], pred_in)
    rmse = np.sqrt(np.mean((pred_in - y[inlier_mask]) ** 2))

    return {
        "params": res1.x,
        "initial_params": res0.x,
        "x": x,
        "y": y,
        "inlier_mask": inlier_mask,
        "r2": r2,
        "rmse": rmse,
        "f_scale": final_scale,
        "initial_mad_scale": f_scale,
        "initial_residual_median": resid_median,
        "n_total": len(x),
        "n_inliers": int(inlier_mask.sum()),
    }


def candidate_row_sets_for_group(group):
    eligible = eligible_rows_for_group(group)
    candidates = []

    def add(label, subset, priority):
        subset = subset.copy().sort_index()
        if len(subset) >= MIN_POINTS and subset["nu_free"].nunique() >= MIN_DISTINCT_NU:
            candidates.append((priority, label, subset))

    exp_only = eligible[eligible["k_obs_method_clean"].eq("exponential")]
    exp_deriv_only = eligible[eligible["k_obs_method_clean"].eq("initial_rate_exp_derivative")]
    preferred = eligible[eligible["k_obs_method_clean"].isin(PREFERRED_METHODS)]
    fallback_only = eligible[eligible["k_obs_method_clean"].eq(FALLBACK_METHOD)]
    preferred_plus_fallback = eligible[eligible["k_obs_method_clean"].isin(PREFERRED_METHODS | {FALLBACK_METHOD})]

    add("exponential only", exp_only, 1)
    add("initial_rate_exp_derivative only", exp_deriv_only, 2)
    add("preferred mixed: exponential + initial_rate_exp_derivative", preferred, 3)
    add("linear fallback only", fallback_only, 4)
    add("preferred plus linear fallback", preferred_plus_fallback, 5)
    add("last-resort mixed methods", eligible, 6)

    if not candidates:
        return eligible, []
    deduped = []
    seen = set()
    for priority, label, subset in candidates:
        key = tuple(subset.index)
        if key in seen:
            continue
        seen.add(key)
        deduped.append((priority, label, subset))
    return eligible, deduped


def evaluate_strategy_a_candidates(group):
    eligible, candidates = candidate_row_sets_for_group(group)
    results = []
    for priority, label, selected in candidates:
        fit = fit_linear_two_step_huber(selected["nu_free"].to_numpy(), selected["k_obs"].to_numpy())
        if fit is None:
            continue
        k0, k2_app = fit["params"]
        inliers = fit["inlier_mask"]
        boots = bayesian_bootstrap_linear(fit["x"][inliers], fit["y"][inliers])
        if boots.size:
            boot_lo, boot_hi = np.percentile(boots, [2.5, 97.5], axis=0)
        else:
            boot_lo = boot_hi = np.array([np.nan, np.nan])
        k2_app_lo, k2_app_hi = boot_lo[1], boot_hi[1]
        k2_app_rel_ci_width = (k2_app_hi - k2_app_lo) / abs(k2_app) if np.isfinite(k2_app) and k2_app != 0 else np.inf
        success = bool(
            np.isfinite(k2_app)
            and np.isfinite(k0)
            and k2_app > 0
            and np.isfinite(k2_app_lo)
            and k2_app_lo > 0
            and np.isfinite(fit["r2"])
            and fit["r2"] >= K2_APP_ACCEPT_MIN_R2
            and np.isfinite(k2_app_rel_ci_width)
            and k2_app_rel_ci_width <= K2_APP_ACCEPT_MAX_REL_CI_WIDTH
        )
        results.append({
            "priority": priority,
            "selection_note": label,
            "selected": selected,
            "fit": fit,
            "bootstrap_params": boots,
            "boot_lo": boot_lo,
            "boot_hi": boot_hi,
            "k2_app_rel_ci_width": k2_app_rel_ci_width,
            "candidate_success": success,
            "candidate_score": (
                int(success),
                int(np.isfinite(k2_app) and k2_app > 0),
                int(np.isfinite(k2_app_lo) and k2_app_lo > 0),
                -k2_app_rel_ci_width if np.isfinite(k2_app_rel_ci_width) else -np.inf,
                fit["r2"] if np.isfinite(fit["r2"]) else -np.inf,
                fit["n_inliers"],
                -priority,
            ),
        })
    if not results:
        return eligible, None, []
    successful = [r for r in results if r["candidate_success"]]
    if successful:
        return eligible, sorted(successful, key=lambda r: r["priority"])[0], results
    return eligible, sorted(results, key=lambda r: r["candidate_score"], reverse=True)[0], results


def bayesian_bootstrap_linear(x, y, n_boot=BOOTSTRAP_N):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    X = np.column_stack([np.ones(len(x)), x])
    out = []
    for _ in range(n_boot):
        weights = rng.dirichlet(np.ones(len(x))) * len(x)
        Xw = X * np.sqrt(weights)[:, None]
        yw = y * np.sqrt(weights)
        try:
            params = np.linalg.lstsq(Xw, yw, rcond=None)[0]
            if np.all(np.isfinite(params)):
                out.append(params)
        except np.linalg.LinAlgError:
            continue
    return np.asarray(out)


def save_kobs_plot(selected, fit, row, out_dir, boot_lo=None, boot_hi=None):
    if not SAVE_KOBS_PLOTS:
        return None
    fig, ax = plt.subplots(figsize=(6.8, 4.9))
    x, y = fit["x"], fit["y"]
    inliers = fit["inlier_mask"]
    eligible_for_plot = row.get("eligible_for_plot")
    if eligible_for_plot is not None and len(eligible_for_plot):
        excluded_for_method = eligible_for_plot.loc[~eligible_for_plot.index.isin(selected.index)]
        x_max = max(x.max(), excluded_for_method["nu_free"].max() if len(excluded_for_method) else x.max())
        extra_y = excluded_for_method["k_obs"].to_numpy(dtype=float) if len(excluded_for_method) else np.array([])
    else:
        excluded_for_method = selected.iloc[0:0]
        x_max = x.max()
        extra_y = np.array([])
    xg = np.linspace(0, x_max * 1.05, 120)
    yg = linear_predict(fit["params"], xg)
    boots = row.get("bootstrap_params")
    boot_pred = np.array([linear_predict(b, xg) for b in boots]) if boots is not None and len(boots) else np.empty((0, len(xg)))
    band_lo = np.percentile(boot_pred, 2.5, axis=0) if len(boot_pred) else np.full_like(xg, np.nan)
    band_hi = np.percentile(boot_pred, 97.5, axis=0) if len(boot_pred) else np.full_like(xg, np.nan)
    selected_y_for_scale = np.concatenate([y, yg, band_hi[np.isfinite(band_hi)], band_lo[np.isfinite(band_lo)]])
    scale, scale_label = scale_for_axis(selected_y_for_scale)

    if len(boot_pred):
        ax.fill_between(xg, band_lo / scale, band_hi / scale, color="#b8d4e0", alpha=0.70, linewidth=0, label="95% bootstrap CI")
    ax.plot(xg, yg / scale, color="#22344a", linewidth=2.6, label="Huber fit")
    if len(excluded_for_method):
        ax.scatter(
            excluded_for_method["nu_free"], excluded_for_method["k_obs"] / scale,
            marker="x", color="0.65", linewidths=1.5, s=52, label="excluded method"
        )
    ax.scatter(x[inliers], y[inliers] / scale, facecolors="white", edgecolors="#22344a", linewidths=1.6, s=48, label="inliers")
    if (~inliers).any():
        ax.scatter(x[~inliers], y[~inliers] / scale, marker="x", color="#c9403a", linewidths=1.8, s=64, label="outliers")
    ax.set_xlim(left=0)
    y_axis_values = selected_y_for_scale[np.isfinite(selected_y_for_scale)] / scale
    if len(y_axis_values):
        y_lo = min(0, np.nanmin(y_axis_values) * 1.08)
        y_hi = np.nanmax(y_axis_values) * 1.12
        if np.isfinite(y_lo) and np.isfinite(y_hi) and y_hi > y_lo:
            ax.set_ylim(y_lo, y_hi)
    ax.set_xlabel("[Nu]$_{free}$ / mol L$^{-1}$")
    ax.set_ylabel("$k_{obs}$" + scale_label + " s$^{-1}$")
    nu_pka = row.get("nu_pka_corr_used", np.nan)
    ax.set_title(
        f"{format_nucleophile_label(row.get('nu_name'), row.get('nu_id'))}: $k_{{obs}}$ vs [Nu]\n"
        f"$pK_a$ = {fmt_float(nu_pka, 2)}, pH = {row['ha_initial_pH']:.2f}, method = {row.get('methods_used', '')}"
    )
    k0, k2_app = fit["params"]
    if boot_lo is None or boot_hi is None:
        k0_ci = k2_app_ci = ("n/a", "n/a")
    else:
        k0_ci = (fmt_sci(boot_lo[0]), fmt_sci(boot_hi[0]))
        k2_app_ci = (fmt_sci(boot_lo[1]), fmt_sci(boot_hi[1]))
    stats_text = (
        f"$k_{{2,\\mathrm{{app}}}}$ = {fmt_sci(k2_app)}\n"
        f"$k_0$ = {fmt_sci(k0)}\n"
        f"R$^2$ = {fmt_float(fit['r2'])}\n"
        f"n = {fit['n_total']} ({fit['n_inliers']} in)\n"
        f"pH$_{{corr}}$ mean = {fmt_float(row.get('pH_corr_ionize_mean', np.nan), 2)}\n"
        f"drift max = {fmt_float(row.get('pH_drift_abs_max', np.nan), 2)}\n"
        f"excluded method rows = {len(excluded_for_method)}\n"
        f"method: {row.get('selection_note', row.get('methods_used', ''))}\n"
        f"95% CI $k_{{2,\\mathrm{{app}}}}$: [{k2_app_ci[0]}, {k2_app_ci[1]}]\n"
        f"95% CI k$_0$: [{k0_ci[0]}, {k0_ci[1]}]\n"
        f"RMSE = {fmt_sci(fit['rmse'])}"
    )
    ax.text(
        0.03, 0.97, stats_text,
        transform=ax.transAxes, va="top", ha="left", fontsize=8.6, color="#22344a",
        bbox=dict(boxstyle="round,pad=.35", facecolor="white", edgecolor="#b8d4e0", alpha=0.92)
    )
    ax.legend(frameon=True, fontsize=8.5, loc="lower right")
    fig.tight_layout()
    path = out_dir / "kobs_vs_nu_plots" / f"kobs_vs_nu_{sanitize_filename(row['nu_id'])}_pH_{row['ha_initial_pH']:.2f}.png"
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


def strategy_a_extract_k2_app(df, out_dir):
    rows = []
    group_cols = ["electrophile_type", "nu_id", "nu_name", "ha_initial_pH"]
    for keys, group in df.groupby(group_cols, dropna=False):
        electrophile, nu_id, nu_name, ha_initial_pH = keys
        eligible, chosen, candidate_results = evaluate_strategy_a_candidates(group)
        selected = chosen["selected"] if chosen is not None else eligible
        selection_note = chosen["selection_note"] if chosen is not None else "insufficient usable points"
        candidate_summary = "; ".join(
            f"{r['selection_note']}: k2_app={r['fit']['params'][1]:.2e}, "
            f"k2_app_lo95={r['boot_lo'][1]:.2e}, relCI={r['k2_app_rel_ci_width']:.2f}, "
            f"R2={r['fit']['r2']:.3f}, n={r['fit']['n_inliers']}/{r['fit']['n_total']}, "
            f"{'pass' if r['candidate_success'] else 'fail'}"
            for r in sorted(candidate_results, key=lambda item: item["priority"])
        )
        base = {
            "electrophile_type": electrophile,
            "nu_id": nu_id,
            "nu_name": nu_name,
            "ha_initial_pH": ha_initial_pH,
            "pH_corr_ionize_mean": selected["pH_corr_ionize"].mean() if len(selected) else np.nan,
            "pH_drift_abs_max": (selected["pH_corr_ionize"] - ha_initial_pH).abs().max() if len(selected) else np.nan,
            "pH_drift_abs_mean": (selected["pH_corr_ionize"] - ha_initial_pH).abs().mean() if len(selected) else np.nan,
            "nu_pka_corr_used": selected["nu_pka_corr"].iloc[0] if len(selected) else np.nan,
            "nu_pka_corr_unique_n": selected["nu_pka_corr"].nunique(dropna=True) if len(selected) else 0,
            "selection_note": selection_note,
            "candidate_summary": candidate_summary,
            "methods_used": ", ".join(sorted(selected["k_obs_method_clean"].dropna().unique())) if len(selected) else "",
            "n_selected": len(selected),
            "n_distinct_nu_free": selected["nu_free"].nunique() if len(selected) else 0,
        }
        if chosen is None or len(selected) < MIN_POINTS or selected["nu_free"].nunique() < MIN_DISTINCT_NU:
            rows.append({**base, "analysis_success": False, "failure_reason": "insufficient points or concentration levels"})
            continue

        fit = chosen["fit"]
        boots = chosen["bootstrap_params"]
        boot_lo = chosen["boot_lo"]
        boot_hi = chosen["boot_hi"]

        k0, k2_app = fit["params"]
        plot_row = {**base, "nu_id": nu_id, "ha_initial_pH": ha_initial_pH}
        plot_row["methods_used"] = base["methods_used"]
        plot_row["selection_note"] = selection_note
        plot_row["bootstrap_params"] = boots
        plot_row["eligible_for_plot"] = eligible
        save_kobs_plot(selected, fit, plot_row, out_dir, boot_lo=boot_lo, boot_hi=boot_hi)
        analysis_success = chosen["candidate_success"]
        rows.append({
            **base,
            "analysis_success": analysis_success,
            "strict_r2_reference_flag": bool(fit["r2"] >= STRICT_R2_REFERENCE_THRESHOLD),
            "failure_reason": "" if analysis_success else "no method subset reached k2_app > 0, k2_app_lo95 > 0, R2 >= 0.50, and relative CI width <= 1.0",
            "k2_app": k2_app,
            "k2_app_lo95": boot_lo[1],
            "k2_app_hi95": boot_hi[1],
            "k2_app_rel_ci_width": chosen["k2_app_rel_ci_width"],
            "k0": k0,
            "k0_lo95": boot_lo[0],
            "k0_hi95": boot_hi[0],
            "r2": fit["r2"],
            "rmse": fit["rmse"],
            "n_total_fit": fit["n_total"],
            "n_inliers": fit["n_inliers"],
            "n_outliers": fit["n_total"] - fit["n_inliers"],
            "outlier_entries": ", ".join(selected.iloc[np.where(~fit["inlier_mask"])[0]]["entry"].astype(str)) if (~fit["inlier_mask"]).any() else "",
        })
    return pd.DataFrame(rows)

## Stage 3: determination of mechanism-specific second-order rate constants

Only successful $k_{2,\mathrm{app}}$ values from the previous step are used here. The corrected reaction pH is represented by the mean `pH_corr_ionize` value of each pH series.

The notebook fits the pH-rate model with fixed electrophile pKa and estimates $k_{2,\mathrm{low}}$ and $k_{2,\mathrm{high}}$. The plot includes:

- Huber inliers and outliers in the pH-rate fit.
- Error bars from the first-stage Bayesian bootstrap of $k_{2,\mathrm{app}}$.
- A bootstrap confidence band for the pH-rate curve.
- The high-pH median analysis using successful $k_{2,\mathrm{app}}$ values where $pH \geq pK_a + 1$.

For some phosphorylating agents, most datasets mainly constrain $k_{2,\mathrm{high}}$; the workbook includes coverage columns indicating whether $k_{2,\mathrm{low}}$ is supported by low- or transition-pH data.

In [ ]:
# ---------------------------------------------------------------------
# Strategy A, second stage: pH-rate fit of k2_app(pH)
# ---------------------------------------------------------------------
def ph_rate_model(params, pH, pKa):
    k2_low, k2_high = params
    a = alpha_deprotonated(pH, pKa)
    return (1.0 - a) * k2_low + a * k2_high


def ph_rate_log_residuals(params, pH, y, pKa, weights=None):
    pred = ph_rate_model(params, pH, pKa)
    resid = safe_log10(pred) - safe_log10(y)
    if weights is not None:
        resid = resid * np.sqrt(weights)
    return resid


def initial_ph_rate_guess(pH, y, pKa):
    pH = np.asarray(pH, dtype=float)
    y = np.asarray(y, dtype=float)
    low_values = y[pH <= pKa]
    high_values = y[pH >= pKa + 1.0]
    k_low0 = np.nanmedian(low_values) if len(low_values) else np.nanpercentile(y, 25)
    k_high0 = np.nanmedian(high_values) if len(high_values) else np.nanpercentile(y, 75)
    tiny = max(np.nanmin(y[y > 0]) * 0.1, 1e-30) if np.any(y > 0) else 1e-12
    return np.maximum([k_low0, k_high0], tiny)


def fit_ph_rate_two_step_huber(pH, y, pKa):
    pH = np.asarray(pH, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = finite_positive_mask(pH, y) & (y > 0)
    pH, y = pH[mask], y[mask]
    if len(y) < MIN_POINTS:
        return None

    x0 = initial_ph_rate_guess(pH, y, pKa)
    bounds = ([1e-30, 1e-30], [np.inf, np.inf])
    res0 = least_squares(
        ph_rate_log_residuals, x0=x0, args=(pH, y, pKa),
        loss="huber", f_scale=1e6, bounds=bounds,
        ftol=1e-12, xtol=1e-12, gtol=1e-12, max_nfev=5000
    )
    resid0 = ph_rate_log_residuals(res0.x, pH, y, pKa)
    mae0 = np.mean(np.abs(resid0))
    f_scale = mae0 if mae0 > 0 else max(np.std(resid0), 1e-3)
    inlier_mask = np.abs(resid0) <= OUTLIER_FACTOR * f_scale
    if inlier_mask.sum() < MIN_POINTS:
        inlier_mask = np.ones(len(y), dtype=bool)

    res1 = least_squares(
        ph_rate_log_residuals, x0=res0.x, args=(pH[inlier_mask], y[inlier_mask], pKa),
        loss="huber", f_scale=f_scale, bounds=bounds,
        ftol=1e-12, xtol=1e-12, gtol=1e-12, max_nfev=5000
    )
    pred_in = ph_rate_model(res1.x, pH[inlier_mask], pKa)
    return {
        "params": res1.x,
        "pH": pH,
        "y": y,
        "inlier_mask": inlier_mask,
        "f_scale": f_scale,
        "r2_log": r2_score(safe_log10(y[inlier_mask]), safe_log10(pred_in)),
        "r2_linear": r2_score(y[inlier_mask], pred_in),
        "rmse_log": np.sqrt(np.mean((safe_log10(pred_in) - safe_log10(y[inlier_mask])) ** 2)),
    }


def bayesian_bootstrap_ph_rate(pH, y, pKa, x0, f_scale, n_boot=BOOTSTRAP_N):
    pH = np.asarray(pH, dtype=float)
    y = np.asarray(y, dtype=float)
    boots = []
    bounds = ([1e-30, 1e-30], [np.inf, np.inf])
    for _ in range(n_boot):
        weights = rng.dirichlet(np.ones(len(y))) * len(y)
        try:
            res = least_squares(
                ph_rate_log_residuals, x0=x0, args=(pH, y, pKa, weights),
                loss="huber", f_scale=f_scale, bounds=bounds,
                ftol=1e-10, xtol=1e-10, gtol=1e-10, max_nfev=3000
            )
            if res.success and np.all(np.isfinite(res.x)):
                boots.append(res.x)
        except Exception:
            continue
    return np.asarray(boots)


def scale_for_axis(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values) & (values > 0)]
    if len(values) == 0:
        return 1.0, ""
    typical = np.nanmedian(values)
    ymax = np.nanmax(values)
    exponent = int(np.floor(np.log10(typical)))
    scale = 10.0 ** exponent
    scaled_max = ymax / scale
    while scaled_max >= 100:
        exponent += 1
        scale = 10.0 ** exponent
        scaled_max = ymax / scale
    while scaled_max < 1:
        exponent -= 1
        scale = 10.0 ** exponent
        scaled_max = ymax / scale
    label = f" / 10$^{{{exponent}}}$" if exponent != 0 else ""
    return scale, label


def plot_ph_rate(profile, fit, boots, plateau, pKa, title, path):
    pH = fit["pH"]
    y = fit["y"]
    inliers = fit["inlier_mask"]
    obs_profile = profile[
        np.isfinite(profile["pH_corr_ionize_mean"])
        & np.isfinite(profile["k2_app"])
        & (profile["k2_app"] > 0)
    ].reset_index(drop=True)
    if len(obs_profile) == len(y) and {"k2_app_lo95", "k2_app_hi95"}.issubset(obs_profile.columns):
        err_low = np.maximum(y - obs_profile["k2_app_lo95"].to_numpy(dtype=float), 0)
        err_high = np.maximum(obs_profile["k2_app_hi95"].to_numpy(dtype=float) - y, 0)
        err_low[~np.isfinite(err_low)] = 0
        err_high[~np.isfinite(err_high)] = 0
    else:
        err_low = np.zeros_like(y)
        err_high = np.zeros_like(y)
    pH_grid = np.linspace(np.nanmin(pH) - 0.2, np.nanmax(pH) + 0.2, 300)
    pred = ph_rate_model(fit["params"], pH_grid, pKa)
    boot_pred = np.array([ph_rate_model(b, pH_grid, pKa) for b in boots]) if len(boots) else np.empty((0, len(pH_grid)))
    band_lo = np.percentile(boot_pred, 2.5, axis=0) if len(boot_pred) else np.full_like(pH_grid, np.nan)
    band_hi = np.percentile(boot_pred, 97.5, axis=0) if len(boot_pred) else np.full_like(pH_grid, np.nan)
    scale, scale_label = scale_for_axis(np.concatenate([y + err_high, pred, band_hi[np.isfinite(band_hi)]]))

    fig, ax = plt.subplots(figsize=(9.2, 4.9))
    ax.fill_between(pH_grid, band_lo / scale, band_hi / scale, color="#9bdfe2", alpha=0.45, linewidth=0, label="95% CI")
    ax.plot(pH_grid, pred / scale, color="#1d3557", linewidth=2.5, label="pH-rate fit")
    ax.errorbar(
        pH[inliers], y[inliers] / scale,
        yerr=np.vstack([err_low[inliers] / scale, err_high[inliers] / scale]),
        fmt="none", ecolor="#457b9d", elinewidth=1.1, capsize=2.5, alpha=0.8, zorder=1
    )
    ax.scatter(pH[inliers], y[inliers] / scale, facecolors="white", edgecolors="#457b9d", linewidths=1.6, s=46, label="inliers")
    if (~inliers).any():
        ax.errorbar(
            pH[~inliers], y[~inliers] / scale,
            yerr=np.vstack([err_low[~inliers] / scale, err_high[~inliers] / scale]),
            fmt="none", ecolor="#e63946", elinewidth=1.1, capsize=2.5, alpha=0.8, zorder=1
        )
        ax.scatter(pH[~inliers], y[~inliers] / scale, marker="D", facecolors="white", edgecolors="#e63946", linewidths=1.6, s=50, label="outliers")
    if np.isfinite(plateau.get("high_pH_k2_app_median", np.nan)):
        ax.axhline(plateau["high_pH_k2_app_median"] / scale, color="#e63946", linestyle="--", linewidth=2, label="median high-pH $k_{2,\\mathrm{app}}$")
        if np.isfinite(plateau.get("high_pH_k2_app_sd", np.nan)):
            lo = (plateau["high_pH_k2_app_median"] - plateau["high_pH_k2_app_sd"]) / scale
            hi = (plateau["high_pH_k2_app_median"] + plateau["high_pH_k2_app_sd"]) / scale
            ax.axhspan(lo, hi, color="#e63946", alpha=0.08)
    ax.axvline(pKa, color="0.7", linestyle=":", linewidth=1.2)
    ax.set_xlabel("mean pH$_{corr,ionize}$")
    ax.set_ylabel("$k_{2,\\mathrm{app}}$" + scale_label + " s$^{-1}$ mol$^{-1}$ L")
    ax.set_title(title)
    if len(boots):
        boot_lo, boot_hi = np.percentile(boots[:, :2], [2.5, 97.5], axis=0)
    else:
        boot_lo = boot_hi = np.array([np.nan, np.nan])
    stats_text = (
        "Optimized parameters\n"
        f"$k_{{2,\\mathrm{{low}}}}$ = {fmt_sci(fit['params'][0])}\n"
        f"95% CI [{fmt_sci(boot_lo[0])}, {fmt_sci(boot_hi[0])}]\n"
        f"$k_{{2,\\mathrm{{high}}}}$ = {fmt_sci(fit['params'][1])}\n"
        f"95% CI [{fmt_sci(boot_lo[1])}, {fmt_sci(boot_hi[1])}]\n\n"
        "Diagnostics\n"
        f"R$^2_{{log}}$ = {fmt_float(fit['r2_log'])}\n"
        f"R$^2_{{linear}}$ = {fmt_float(fit['r2_linear'])}\n"
        f"RMSE$_{{log}}$ = {fmt_float(fit['rmse_log'])}\n"
        f"inliers/total = {int(inliers.sum())}/{len(inliers)}\n"
        f"pKa = {fmt_float(pKa, 2)}\n\n"
        "High-pH $k_{2,\\mathrm{app}}$\n"
        f"median = {fmt_sci(plateau.get('high_pH_k2_app_median', np.nan))}\n"
        f"mean = {fmt_sci(plateau.get('high_pH_k2_app_mean', np.nan))}\n"
        f"SD = {fmt_sci(plateau.get('high_pH_k2_app_sd', np.nan))}\n"
        f"n = {plateau.get('high_pH_k2_app_n', 0)}"
    )
    ax.text(
        1.03, 0.02, stats_text,
        transform=ax.transAxes, va="bottom", fontsize=9.2,
        bbox=dict(facecolor="white", edgecolor="0.85", alpha=0.88)
    )
    ax.legend(frameon=False, loc="upper right")
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


def strategy_a_ph_rate(k2_app_summary, out_dir):
    successful = k2_app_summary[
        k2_app_summary["analysis_success"].fillna(False)
        & np.isfinite(k2_app_summary["k2_app"])
        & (k2_app_summary["k2_app"] > 0)
    ].copy()
    rows = []
    for (electrophile, nu_id, nu_name), profile in successful.groupby(["electrophile_type", "nu_id", "nu_name"], dropna=False):
        pKa = P_ELECTROPHILE_PKA.get(str(electrophile), np.nan)
        pH = profile["pH_corr_ionize_mean"].to_numpy(dtype=float)
        y = profile["k2_app"].to_numpy(dtype=float)
        base = {
            "electrophile_type": electrophile,
            "nu_id": nu_id,
            "nu_name": nu_name,
            "electrophile_pKa": pKa,
            "n_successful_k2_app": len(profile),
        }
        high = profile[profile["pH_corr_ionize_mean"] >= pKa + 1.0] if np.isfinite(pKa) else profile.iloc[0:0]
        low_region = profile[profile["pH_corr_ionize_mean"] <= pKa - 1.0] if np.isfinite(pKa) else profile.iloc[0:0]
        transition_region = profile[
            (profile["pH_corr_ionize_mean"] > pKa - 1.0)
            & (profile["pH_corr_ionize_mean"] < pKa + 1.0)
        ] if np.isfinite(pKa) else profile.iloc[0:0]
        plateau = {
            "high_pH_k2_app_mean": high["k2_app"].mean() if len(high) else np.nan,
            "high_pH_k2_app_median": high["k2_app"].median() if len(high) else np.nan,
            "high_pH_k2_app_sd": high["k2_app"].std(ddof=1) if len(high) > 1 else np.nan,
            "high_pH_k2_app_n": len(high),
            "n_low_region_points": len(low_region),
            "n_transition_region_points": len(transition_region),
            "n_high_region_points": len(high),
            "k2_low_constrained_by_data": len(low_region) + len(transition_region) >= MIN_POINTS,
        }
        if not np.isfinite(pKa) or len(profile) < MIN_POINTS:
            rows.append({**base, **plateau, "fit_success": False, "failure_reason": "missing pKa or fewer than three successful k2_app values"})
            continue

        fit = fit_ph_rate_two_step_huber(pH, y, pKa)
        if fit is None:
            rows.append({**base, **plateau, "fit_success": False, "failure_reason": "pH-rate fit failed"})
            continue
        inliers = fit["inlier_mask"]
        boots = bayesian_bootstrap_ph_rate(fit["pH"][inliers], fit["y"][inliers], pKa, fit["params"], fit["f_scale"])
        if len(boots):
            boot_lo, boot_hi = np.percentile(boots, [2.5, 97.5], axis=0)
        else:
            boot_lo = boot_hi = np.array([np.nan, np.nan])
        plot_path = out_dir / "plots" / f"strategy_A_ph_rate_{sanitize_filename(nu_id)}_{sanitize_filename(electrophile)}.png"
        plot_ph_rate(profile, fit, boots, plateau, pKa, f"{format_nucleophile_label(nu_name, nu_id)} – {electrophile}", plot_path)
        rows.append({
            **base,
            **plateau,
            "fit_success": True,
            "failure_reason": "",
            "k2_low": fit["params"][0],
            "k2_low_lo95": boot_lo[0],
            "k2_low_hi95": boot_hi[0],
            "k2_high": fit["params"][1],
            "k2_high_lo95": boot_lo[1],
            "k2_high_hi95": boot_hi[1],
            "log10_k2_low": np.log10(fit["params"][0]),
            "log10_k2_high": np.log10(fit["params"][1]),
            "r2_log": fit["r2_log"],
            "r2_linear": fit["r2_linear"],
            "rmse_log": fit["rmse_log"],
            "n_fit_total": len(fit["y"]),
            "n_fit_inliers": int(inliers.sum()),
            "n_fit_outliers": int((~inliers).sum()),
            "outlier_pH_values": ", ".join(f"{v:.3g}" for v in fit["pH"][~inliers]) if (~inliers).any() else "",
            "plot_path": str(plot_path),
        })
    return pd.DataFrame(rows)

## Stage 4: direct global fitting of observed rate constants

Strategy B fits the selected $k_{\mathrm{obs}}$ values directly instead of first reducing them to separate $k_{2,\mathrm{app}}$ estimates. The model is:

$$
k_{obs,i} = k_{0,pH(i)} + [Nu]_{free,i} \left((1-\alpha_i)k_{2,\mathrm{low}} + \alpha_i k_{2,\mathrm{high}}\right)
$$

where $k_0$ is fitted separately for each pH series. This strategy can be more stable than Strategy A when individual pH-series regressions are sparse or noisy.

The outputs include two plot types:

- observed-vs-predicted $k_{\mathrm{obs}}$ diagnostic plots in logarithmic space;
- global pH-rate profile plots showing the fitted $k_{2,\mathrm{app}}(pH)$ curve with a bootstrap confidence band and Strategy A points overlaid when available.

In [ ]:
# ---------------------------------------------------------------------
# Strategy B: direct global kobs fit with pH-dependent k2,app and pH-specific k0
# ---------------------------------------------------------------------
def select_rows_for_strategy_b(df):
    parts = []
    notes = []
    group_cols = ["electrophile_type", "nu_id", "nu_name", "ha_initial_pH"]
    for keys, group in df.groupby(group_cols, dropna=False):
        selected, note = select_rows_for_group(group)
        if len(selected) >= MIN_POINTS and selected["nu_free"].nunique() >= MIN_DISTINCT_NU:
            tmp = selected.copy()
            tmp["strategy_b_selection_note"] = note
            parts.append(tmp)
            notes.append((*keys, note, len(tmp)))
    if not parts:
        return pd.DataFrame(columns=df.columns), pd.DataFrame(notes)
    notes_df = pd.DataFrame(notes, columns=["electrophile_type", "nu_id", "nu_name", "ha_initial_pH", "selection_note", "n_selected"])
    return pd.concat(parts, axis=0).sort_index(), notes_df


def global_predict(params, nu_free, pH, group_idx, pKa):
    k2_low, k2_high = params[:2]
    k0_values = params[2:]
    k2_app_eff = ph_rate_model([k2_low, k2_high], pH, pKa)
    return k0_values[group_idx] + nu_free * k2_app_eff


def global_log_residuals(params, nu_free, pH, group_idx, y, pKa, weights=None):
    pred = global_predict(params, nu_free, pH, group_idx, pKa)
    resid = safe_log10(pred) - safe_log10(y)
    if weights is not None:
        resid = resid * np.sqrt(weights)
    return resid


def fit_global_two_step_huber(profile, pKa, strategy_a_row=None):
    profile = profile.copy()
    profile["pH_group_mean"] = profile.groupby("ha_initial_pH")["pH_corr_ionize"].transform("mean")
    groups = pd.Index(sorted(profile["ha_initial_pH"].dropna().unique()))
    group_ph_means = profile.groupby("ha_initial_pH")["pH_corr_ionize"].mean().reindex(groups).to_numpy(dtype=float)
    group_map = {g: i for i, g in enumerate(groups)}
    group_idx = profile["ha_initial_pH"].map(group_map).to_numpy(dtype=int)
    nu = profile["nu_free"].to_numpy(dtype=float)
    pH = profile["pH_group_mean"].to_numpy(dtype=float)
    y = profile["k_obs"].to_numpy(dtype=float)
    mask = finite_positive_mask(nu, pH, y) & (nu > 0) & (y > 0)
    nu, pH, y, group_idx = nu[mask], pH[mask], y[mask], group_idx[mask]
    if len(y) < MIN_POINTS or len(np.unique(group_idx)) < 1:
        return None

    if strategy_a_row is not None and bool(strategy_a_row.get("fit_success", False)):
        k2_low0 = max(float(strategy_a_row.get("k2_low", np.nan)), 1e-30)
        k2_high0 = max(float(strategy_a_row.get("k2_high", np.nan)), 1e-30)
    else:
        k2_low0 = max(np.nanpercentile(y / np.maximum(nu, 1e-30), 25), 1e-30)
        k2_high0 = max(np.nanpercentile(y / np.maximum(nu, 1e-30), 75), 1e-30)
    k0_init = []
    for gi in range(len(groups)):
        vals = y[group_idx == gi]
        k0_init.append(max(np.nanpercentile(vals, 10), 1e-30))
    x0 = np.array([k2_low0, k2_high0, *k0_init], dtype=float)
    bounds = (np.full_like(x0, 1e-30), np.full_like(x0, np.inf))

    res0 = least_squares(
        global_log_residuals, x0=x0, args=(nu, pH, group_idx, y, pKa),
        loss="huber", f_scale=1e6, bounds=bounds,
        ftol=1e-11, xtol=1e-11, gtol=1e-11, max_nfev=10000
    )
    resid0 = global_log_residuals(res0.x, nu, pH, group_idx, y, pKa)
    mae0 = np.mean(np.abs(resid0))
    f_scale = mae0 if mae0 > 0 else max(np.std(resid0), 1e-3)
    inlier_mask = np.abs(resid0) <= OUTLIER_FACTOR * f_scale
    if inlier_mask.sum() < max(MIN_POINTS, len(x0)):
        inlier_mask = np.ones(len(y), dtype=bool)

    res1 = least_squares(
        global_log_residuals, x0=res0.x,
        args=(nu[inlier_mask], pH[inlier_mask], group_idx[inlier_mask], y[inlier_mask], pKa),
        loss="huber", f_scale=f_scale, bounds=bounds,
        ftol=1e-11, xtol=1e-11, gtol=1e-11, max_nfev=10000
    )
    pred_in = global_predict(res1.x, nu[inlier_mask], pH[inlier_mask], group_idx[inlier_mask], pKa)
    group_has_inlier = np.array([np.any(inlier_mask & (group_idx == gi)) for gi in range(len(groups))], dtype=bool)
    params_report = res1.x.copy()
    if len(params_report) >= 2 + len(group_has_inlier):
        params_report[2:][~group_has_inlier] = np.nan
    return {
        "params": res1.x,
        "params_report": params_report,
        "groups": groups,
        "group_ph_means": group_ph_means,
        "group_has_inlier": group_has_inlier,
        "nu_free": nu,
        "pH": pH,
        "group_idx": group_idx,
        "y": y,
        "inlier_mask": inlier_mask,
        "f_scale": f_scale,
        "r2_log": r2_score(safe_log10(y[inlier_mask]), safe_log10(pred_in)),
        "rmse_log": np.sqrt(np.mean((safe_log10(pred_in) - safe_log10(y[inlier_mask])) ** 2)),
    }


def bayesian_bootstrap_global(fit, pKa, n_boot=BOOTSTRAP_N):
    idx = fit["inlier_mask"]
    nu = fit["nu_free"][idx]
    pH = fit["pH"][idx]
    group_idx = fit["group_idx"][idx]
    y = fit["y"][idx]
    x0 = fit["params"]
    bounds = (np.full_like(x0, 1e-30), np.full_like(x0, np.inf))
    boots = []
    for _ in range(n_boot):
        weights = rng.dirichlet(np.ones(len(y))) * len(y)
        try:
            res = least_squares(
                global_log_residuals, x0=x0, args=(nu, pH, group_idx, y, pKa, weights),
                loss="huber", f_scale=fit["f_scale"], bounds=bounds,
                ftol=1e-9, xtol=1e-9, gtol=1e-9, max_nfev=5000
            )
            if res.success and np.all(np.isfinite(res.x[:2])):
                boots.append(res.x)
        except Exception:
            continue
    return np.asarray(boots)


def plot_global_obs_pred(fit, pKa, title, path, boots=None):
    y = fit["y"]
    pred = global_predict(fit.get("params_report", fit["params"]), fit["nu_free"], fit["pH"], fit["group_idx"], pKa)
    inliers = fit["inlier_mask"]
    finite_plot = np.isfinite(y) & np.isfinite(pred) & (y > 0) & (pred > 0)
    inlier_plot = inliers & finite_plot
    outlier_plot = (~inliers) & finite_plot
    outlier_unconstrained = (~inliers) & (~finite_plot)
    fig, ax = plt.subplots(figsize=(8.2, 4.8))
    ax.scatter(safe_log10(y[inlier_plot]), safe_log10(pred[inlier_plot]), facecolors="white", edgecolors="#457b9d", linewidths=1.5, label="inliers")
    if outlier_plot.any():
        ax.scatter(safe_log10(y[outlier_plot]), safe_log10(pred[outlier_plot]), marker="D", facecolors="white", edgecolors="#e63946", linewidths=1.5, label="outliers")
    lims = [
        min(np.nanmin(safe_log10(y[finite_plot])), np.nanmin(safe_log10(pred[finite_plot]))),
        max(np.nanmax(safe_log10(y[finite_plot])), np.nanmax(safe_log10(pred[finite_plot]))),
    ]
    ax.plot(lims, lims, color="black", linewidth=1.5)
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel("log$_{10}$(observed $k_{obs}$)")
    ax.set_ylabel("log$_{10}$(predicted $k_{obs}$)")
    ax.set_title(title)
    if boots is not None and len(boots):
        boot_lo, boot_hi = np.percentile(boots[:, :2], [2.5, 97.5], axis=0)
    else:
        boot_lo = boot_hi = np.array([np.nan, np.nan])
    stats_text = (
        "Global fit parameters\n"
        f"$k_{{2,\\mathrm{{low}}}}$ = {fmt_sci(fit['params'][0])}\n"
        f"95% CI [{fmt_sci(boot_lo[0])}, {fmt_sci(boot_hi[0])}]\n"
        f"$k_{{2,\\mathrm{{high}}}}$ = {fmt_sci(fit['params'][1])}\n"
        f"95% CI [{fmt_sci(boot_lo[1])}, {fmt_sci(boot_hi[1])}]\n\n"
        "Diagnostics\n"
        f"R$^2_{{log}}$ = {fmt_float(fit['r2_log'])}\n"
        f"RMSE$_{{log}}$ = {fmt_float(fit['rmse_log'])}\n"
        f"inliers/total = {int(inliers.sum())}/{len(inliers)}\n"
        f"pH-specific $k_0$ values = {int(np.sum(fit.get('group_has_inlier', np.ones(len(fit['groups']), dtype=bool))))}/{len(fit['groups'])}\n"
        f"outliers with unconstrained $k_0$ = {int(outlier_unconstrained.sum())}\n"
        f"pKa = {fmt_float(pKa, 2)}"
    )
    ax.text(
        1.03, 0.02, stats_text,
        transform=ax.transAxes, va="bottom", fontsize=9.2,
        bbox=dict(facecolor="white", edgecolor="0.85", alpha=0.88)
    )
    ax.legend(frameon=False, loc="best")
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


def plot_global_ph_rate(fit, pKa, title, path, boots=None, strategy_a_profile=None):
    pH_min = np.nanmin(fit["pH"]) - 0.2
    pH_max = np.nanmax(fit["pH"]) + 0.2
    pH_grid = np.linspace(pH_min, pH_max, 300)
    k2_app_grid = ph_rate_model(fit["params"][:2], pH_grid, pKa)
    if boots is not None and len(boots):
        boot_pred = np.array([ph_rate_model(b[:2], pH_grid, pKa) for b in boots])
        band_lo = np.percentile(boot_pred, 2.5, axis=0)
        band_hi = np.percentile(boot_pred, 97.5, axis=0)
    else:
        band_lo = band_hi = np.full_like(pH_grid, np.nan)

    overlay = None
    overlay_ymax = np.array([])
    if strategy_a_profile is not None and len(strategy_a_profile):
        overlay = strategy_a_profile[
            strategy_a_profile["analysis_success"].fillna(False)
            & np.isfinite(strategy_a_profile["k2_app"])
            & (strategy_a_profile["k2_app"] > 0)
        ].copy()
        if len(overlay):
            overlay_y = overlay["k2_app"].to_numpy(dtype=float)
            overlay_hi = overlay["k2_app_hi95"].to_numpy(dtype=float) if "k2_app_hi95" in overlay else overlay_y
            overlay_ymax = np.concatenate([overlay_y[np.isfinite(overlay_y)], overlay_hi[np.isfinite(overlay_hi)]])

    scale, scale_label = scale_for_axis(np.concatenate([k2_app_grid, band_hi[np.isfinite(band_hi)], overlay_ymax]))
    fig, ax = plt.subplots(figsize=(9.2, 4.9))
    if np.any(np.isfinite(band_lo)) and np.any(np.isfinite(band_hi)):
        ax.fill_between(pH_grid, band_lo / scale, band_hi / scale, color="#9bdfe2", alpha=0.45, linewidth=0, label="95% CI")
    ax.plot(pH_grid, k2_app_grid / scale, color="#1d3557", linewidth=2.5, label="global pH-rate profile")
    if overlay is not None and len(overlay):
        pH_pts = overlay["pH_corr_ionize_mean"].to_numpy(dtype=float)
        y_pts = overlay["k2_app"].to_numpy(dtype=float)
        lo = overlay["k2_app_lo95"].to_numpy(dtype=float) if "k2_app_lo95" in overlay else y_pts
        hi = overlay["k2_app_hi95"].to_numpy(dtype=float) if "k2_app_hi95" in overlay else y_pts
        err_low = np.maximum(y_pts - lo, 0)
        err_high = np.maximum(hi - y_pts, 0)
        err_low[~np.isfinite(err_low)] = 0
        err_high[~np.isfinite(err_high)] = 0
        ax.errorbar(
            pH_pts, y_pts / scale,
            yerr=np.vstack([err_low / scale, err_high / scale]),
            fmt="none", ecolor="#457b9d", elinewidth=1.1, capsize=2.5, alpha=0.8
        )
        ax.scatter(pH_pts, y_pts / scale, facecolors="white", edgecolors="#457b9d", linewidths=1.6, s=46, label="Strategy A $k_{2,\\mathrm{app}}$")
    ax.axvline(pKa, color="0.7", linestyle=":", linewidth=1.2)
    ax.set_xlabel("mean pH$_{corr,ionize}$")
    ax.set_ylabel("$k_{2,\\mathrm{app}}$" + scale_label + " s$^{-1}$ mol$^{-1}$ L")
    ax.set_title(title)
    if boots is not None and len(boots):
        boot_lo, boot_hi = np.percentile(boots[:, :2], [2.5, 97.5], axis=0)
    else:
        boot_lo = boot_hi = np.array([np.nan, np.nan])
    stats_text = (
        "Global pH-rate parameters\n"
        f"$k_{{2,\\mathrm{{low}}}}$ = {fmt_sci(fit['params'][0])}\n"
        f"95% CI [{fmt_sci(boot_lo[0])}, {fmt_sci(boot_hi[0])}]\n"
        f"$k_{{2,\\mathrm{{high}}}}$ = {fmt_sci(fit['params'][1])}\n"
        f"95% CI [{fmt_sci(boot_lo[1])}, {fmt_sci(boot_hi[1])}]\n\n"
        f"R$^2_{{log}}$ = {fmt_float(fit['r2_log'])}\n"
        f"RMSE$_{{log}}$ = {fmt_float(fit['rmse_log'])}\n"
        f"inliers/total = {int(fit['inlier_mask'].sum())}/{len(fit['inlier_mask'])}\n"
        f"pH-specific $k_0$ values = {len(fit['groups'])}\n"
        f"pKa = {fmt_float(pKa, 2)}"
    )
    ax.text(
        1.03, 0.02, stats_text,
        transform=ax.transAxes, va="bottom", fontsize=9.2,
        bbox=dict(facecolor="white", edgecolor="0.85", alpha=0.88)
    )
    ax.legend(frameon=False, loc="best")
    fig.tight_layout()
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)


def strategy_b_global(df, ph_rate_summary, k2_app_summary, out_dir):
    selected, selection_notes = select_rows_for_strategy_b(df)
    rows = []
    k0_rows = []
    for (electrophile, nu_id, nu_name), profile in selected.groupby(["electrophile_type", "nu_id", "nu_name"], dropna=False):
        pKa = P_ELECTROPHILE_PKA.get(str(electrophile), np.nan)
        base = {
            "electrophile_type": electrophile,
            "nu_id": nu_id,
            "nu_name": nu_name,
            "electrophile_pKa": pKa,
            "n_selected_kobs": len(profile),
            "n_pH_groups": profile["ha_initial_pH"].nunique(),
            "n_low_region_rows": int((profile["pH_corr_ionize"] <= pKa - 1.0).sum()) if np.isfinite(pKa) else 0,
            "n_transition_region_rows": int(((profile["pH_corr_ionize"] > pKa - 1.0) & (profile["pH_corr_ionize"] < pKa + 1.0)).sum()) if np.isfinite(pKa) else 0,
            "n_high_region_rows": int((profile["pH_corr_ionize"] >= pKa + 1.0).sum()) if np.isfinite(pKa) else 0,
        }
        strategy_a_match = ph_rate_summary[
            (ph_rate_summary["electrophile_type"] == electrophile)
            & (ph_rate_summary["nu_id"] == nu_id)
        ]
        strategy_a_row = strategy_a_match.iloc[0].to_dict() if len(strategy_a_match) else None
        if not np.isfinite(pKa) or len(profile) < MIN_POINTS:
            rows.append({**base, "fit_success": False, "failure_reason": "missing pKa or too few selected kobs rows"})
            continue
        fit = fit_global_two_step_huber(profile, pKa, strategy_a_row=strategy_a_row)
        if fit is None:
            rows.append({**base, "fit_success": False, "failure_reason": "global fit failed"})
            continue
        boots = bayesian_bootstrap_global(fit, pKa)
        if len(boots):
            boot_lo, boot_hi = np.percentile(boots[:, :2], [2.5, 97.5], axis=0)
        else:
            boot_lo = boot_hi = np.array([np.nan, np.nan])
        plot_path = out_dir / "plots" / f"strategy_B_global_obs_pred_{sanitize_filename(nu_id)}_{sanitize_filename(electrophile)}.png"
        plot_global_obs_pred(fit, pKa, f"Global fit: {format_nucleophile_label(nu_name, nu_id)} – {electrophile}", plot_path, boots=boots)
        global_profile_path = out_dir / "plots" / f"strategy_B_global_ph_rate_{sanitize_filename(nu_id)}_{sanitize_filename(electrophile)}.png"
        strategy_a_profile = k2_app_summary[
            (k2_app_summary["electrophile_type"] == electrophile)
            & (k2_app_summary["nu_id"] == nu_id)
        ]
        plot_global_ph_rate(
            fit, pKa, f"Global pH-rate profile: {format_nucleophile_label(nu_name, nu_id)} – {electrophile}",
            global_profile_path, boots=boots, strategy_a_profile=strategy_a_profile
        )
        rows.append({
            **base,
            "fit_success": True,
            "failure_reason": "",
            "k2_low": fit["params"][0],
            "k2_low_lo95": boot_lo[0],
            "k2_low_hi95": boot_hi[0],
            "k2_high": fit["params"][1],
            "k2_high_lo95": boot_lo[1],
            "k2_high_hi95": boot_hi[1],
            "log10_k2_low": np.log10(fit["params"][0]),
            "log10_k2_high": np.log10(fit["params"][1]),
            "r2_log": fit["r2_log"],
            "rmse_log": fit["rmse_log"],
            "n_fit_total": len(fit["y"]),
            "n_fit_inliers": int(fit["inlier_mask"].sum()),
            "n_fit_outliers": int((~fit["inlier_mask"]).sum()),
            "plot_path": str(plot_path),
            "ph_rate_profile_plot_path": str(global_profile_path),
        })
        for i, pH_group in enumerate(fit["groups"]):
            k0_constrained = bool(fit.get("group_has_inlier", np.ones(len(fit["groups"]), dtype=bool))[i])
            k0 = fit.get("params_report", fit["params"])[2 + i]
            k0_rows.append({
                "electrophile_type": electrophile,
                "nu_id": nu_id,
                "nu_name": nu_name,
                "ha_initial_pH": pH_group,
                "pH_corr_ionize_mean": fit["group_ph_means"][i],
                "k0_global": k0,
                "k0_constrained_by_inliers": k0_constrained,
            })
    return pd.DataFrame(rows), pd.DataFrame(k0_rows), selection_notes

## Run analysis and write outputs

For each dataset listed in `DATASET_DIRS`, the notebook writes:

- `strategy_A_kobs_to_k2_app`: concise per-pH $k_{\mathrm{obs}}$ versus $[\mathrm{Nu}]_{\mathrm{free}}$ fit results.
- `strategy_A_selection_audit`: method-subset and candidate-selection details for auditability.
- `strategy_A_ph_rate`: pH-rate fit summaries and high-pH median analysis.
- `strategy_B_global`: global-fit parameter summaries and plot paths.
- `strategy_B_k0_by_pH`: pH-specific background rates from the global fit, including mean corrected pH.
- `electrophile_pKa`: pKa values used in the analysis.

PNG plots are saved in two folders: `plots` for pH-rate/global diagnostics and `kobs_vs_nu_plots` for first-stage Strategy A regressions.

In [ ]:
# ---------------------------------------------------------------------
# Run one dataset and write outputs
# ---------------------------------------------------------------------
def analyze_dataset(dataset_name):
    dataset_dir = BASE_DIR / dataset_name
    out_dir = output_dir_for(dataset_dir)
    df, fulldata = load_dataset(dataset_dir)
    print(f"\n=== {dataset_name} ===")
    print(f"Input: {fulldata.name}; rows={len(df)}; output={out_dir}")

    k2_app_summary = strategy_a_extract_k2_app(df, out_dir)
    ph_rate_summary = strategy_a_ph_rate(k2_app_summary, out_dir)
    global_summary, global_k0, strategy_b_selection = strategy_b_global(df, ph_rate_summary, k2_app_summary, out_dir)

    strategy_a_cols = [
        "electrophile_type", "nu_id", "nu_name", "ha_initial_pH", "pH_corr_ionize_mean",
        "pH_drift_abs_max", "selection_note", "n_selected", "n_inliers", "n_outliers",
        "outlier_entries", "k2_app", "k2_app_lo95", "k2_app_hi95", "k2_app_rel_ci_width", "k0", "k0_lo95", "k0_hi95",
        "r2", "rmse", "analysis_success", "failure_reason",
    ]
    strategy_a_rename = {
        "pH_drift_abs_max": "pH_corr_ionize_abs_drift_max",
        "selection_note": "k_obs_values_used",
    }
    strategy_a_main = k2_app_summary[[c for c in strategy_a_cols if c in k2_app_summary.columns]].rename(columns=strategy_a_rename)

    strategy_a_audit_cols = [
        "electrophile_type", "nu_id", "ha_initial_pH", "selection_note", "methods_used",
        "candidate_summary", "n_distinct_nu_free", "pH_drift_abs_mean",
        "strict_r2_reference_flag", "outlier_entries", "failure_reason",
    ]
    strategy_a_audit = k2_app_summary[[c for c in strategy_a_audit_cols if c in k2_app_summary.columns]].rename(columns={
        "selection_note": "selected_candidate",
        "methods_used": "raw_k_obs_sources_in_selected_fit",
        "pH_drift_abs_mean": "pH_corr_ionize_abs_drift_mean",
        "strict_r2_reference_flag": "strict_r2_ge_0_90",
    })

    ph_rate_cols = [
        "electrophile_type", "nu_id", "nu_name", "electrophile_pKa", "n_successful_k2_app",
        "n_low_region_points", "n_transition_region_points", "n_high_region_points",
        "k2_low_constrained_by_data", "k2_low", "k2_low_lo95", "k2_low_hi95",
        "k2_high", "k2_high_lo95", "k2_high_hi95", "high_pH_k2_app_mean", "high_pH_k2_app_median",
        "high_pH_k2_app_sd", "high_pH_k2_app_n", "r2_log", "rmse_log", "fit_success", "failure_reason",
        "plot_path",
    ]
    ph_rate_main = ph_rate_summary[[c for c in ph_rate_cols if c in ph_rate_summary.columns]]

    global_cols = [
        "electrophile_type", "nu_id", "nu_name", "electrophile_pKa", "n_selected_kobs",
        "n_pH_groups", "n_low_region_rows", "n_transition_region_rows", "n_high_region_rows",
        "k2_low", "k2_low_lo95", "k2_low_hi95", "k2_high", "k2_high_lo95", "k2_high_hi95",
        "r2_log", "rmse_log", "n_fit_inliers", "n_fit_outliers", "fit_success",
        "failure_reason", "plot_path", "ph_rate_profile_plot_path",
    ]
    global_main = global_summary[[c for c in global_cols if c in global_summary.columns]].rename(columns={
        "n_selected_kobs": "n_selected_k_obs",
    })
    global_k0_main = global_k0.copy()

    excel_path = out_dir / f"{dataset_name}_k2_next_step_analysis.xlsx"
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
        strategy_a_main.to_excel(writer, sheet_name="strategy_A_kobs_to_k2_app", index=False)
        strategy_a_audit.to_excel(writer, sheet_name="strategy_A_selection_audit", index=False)
        ph_rate_main.to_excel(writer, sheet_name="strategy_A_ph_rate", index=False)
        global_main.to_excel(writer, sheet_name="strategy_B_global", index=False)
        global_k0_main.to_excel(writer, sheet_name="strategy_B_k0_by_pH", index=False)
        pka_df = pd.DataFrame({"electrophile_type": list(P_ELECTROPHILE_PKA.keys()), "pKa": list(P_ELECTROPHILE_PKA.values())})
        pka_df.to_excel(writer, sheet_name="electrophile_pKa", index=False)
        for ws in writer.book.worksheets:
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions
            for column_cells in ws.columns:
                max_len = max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells)
                width = min(max(max_len + 2, 10), 42)
                ws.column_dimensions[column_cells[0].column_letter].width = width

    print(f"Strategy A kobs -> k2,app: {int(k2_app_summary['analysis_success'].fillna(False).sum())}/{len(k2_app_summary)} successful groups")
    print(f"Strategy A pH-rate: {int(ph_rate_summary['fit_success'].fillna(False).sum())}/{len(ph_rate_summary)} successful profiles")
    print(f"Strategy B global: {int(global_summary['fit_success'].fillna(False).sum())}/{len(global_summary)} successful profiles")
    print(f"Wrote {excel_path}")
    return {
        "dataset": dataset_name,
        "output_dir": out_dir,
        "excel_path": excel_path,
        "strategy_A_kobs_to_k2_app": k2_app_summary,
        "strategy_A_ph_rate": ph_rate_summary,
        "strategy_B_global": global_summary,
        "strategy_B_k0": global_k0,
    }


all_results = {}
for dataset_name in DATASET_DIRS:
    all_results[dataset_name] = analyze_dataset(dataset_name)

summary_rows = []
for dataset_name, result in all_results.items():
    summary_rows.append({
        "dataset": dataset_name,
        "excel_path": str(result["excel_path"]),
        "strategy_A_successful_k2_app": int(result["strategy_A_kobs_to_k2_app"]["analysis_success"].fillna(False).sum()),
        "strategy_A_total_k2_app_groups": len(result["strategy_A_kobs_to_k2_app"]),
        "strategy_A_successful_ph_rate": int(result["strategy_A_ph_rate"]["fit_success"].fillna(False).sum()),
        "strategy_B_successful_global": int(result["strategy_B_global"]["fit_success"].fillna(False).sum()),
    })
run_summary = pd.DataFrame(summary_rows)
run_summary